<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Operaciones de Aprendizaje Automático III**
# **Clase 8, RFT (Rejection Sampling Fine-tuning)**

* en este notebook el modelo lee un problema en español y escribe una expresion aritmética que lo resuelve
* el juez la calcula y la compara con la respuesta correcta:

```
problema:   Ana compra 4 cajas con 12 naranjas cada una y regala 7. ¿Cuántas le quedan?
modelo: 4 * 12 - 7 -> 41   OK
modelo: 4 + 12 - 7 -> 9   NO OK
```

el modelo ya sabe resolver más de lo que muestra en un solo intento


In [1]:
# colab trae un torchao antiguo que choca con peft, no se usa, así que se quita
!pip uninstall -y -q torchao
!pip install -q peft==0.21.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 18.1 MB/s eta 0:00:00


**1. Los problemas**

* se generan con plantillas
* cada plantilla es un tipo de problema con su formula
* los números cambian en cada problema

In [2]:
import random
random.seed(0)

In [3]:
plantillas = {
    "multiplicar y restar": ("Ana compra {a} cajas con {b} naranjas cada una y regala {c}. ¿Cuántas naranjas le quedan?",
                             lambda a, b, c: a * b - c),
    "restar y sumar": ("Un minibús lleva {a} pasajeros. En la parada bajan {b} y suben {c}. ¿Cuántos pasajeros hay ahora?",
                       lambda a, b, c: a - b + c),
    "dinero": ("Luis tiene {a} bolivianos y compra {b} salteñas a {c} bolivianos cada una. ¿Cuánto dinero le queda?",
               lambda a, b, c: a - b * c),
    "sillas": ("En un aula hay {a} filas de {b} sillas y {c} sillas están ocupadas. ¿Cuántas sillas libres hay?",
               lambda a, b, c: a * b - c),
    "lectura": ("María lee {a} páginas por día durante {b} días y le faltan {c} páginas para terminar. ¿Cuántas páginas tiene el libro?",
                lambda a, b, c: a * b + c),
    "combustible": ("Un bus sale con {a} litros de combustible y gasta {b} litros por hora durante {c} horas. ¿Cuántos litros le quedan?",
                    lambda a, b, c: a - b * c),
}

In [4]:
def crear_problemas(por_tipo=20):
    problemas = []
    for tipo, (texto, formula) in plantillas.items():
        for _ in range(por_tipo):
            a, b, c = random.randint(20, 90), random.randint(2, 9), random.randint(2, 9)
            if formula(a, b, c) > 0:  # sin respuestas negativas
                problemas.append({"tipo": tipo, "pregunta": texto.format(a=a, b=b, c=c), "respuesta": formula(a, b, c)})
    return problemas

In [5]:
problemas = crear_problemas()
random.shuffle(problemas)

* 5 problemas de cada tipo para prueba, el resto para entrenar
* prueba y entrenamiento usan las mismas plantillas, solo cambian los números
* la evaluación mide si el modelo aprendió estos seis tipos

In [6]:
prueba = [p for t in plantillas for p in [q for q in problemas if q["tipo"] == t][:5]]
entrenamiento = [p for p in problemas if p not in prueba]
print("entrenamiento:", len(entrenamiento), ", prueba:", len(prueba))
print(prueba[0])

entrenamiento: 79 , prueba: 30
{'tipo': 'multiplicar y restar', 'pregunta': 'Ana compra 85 cajas con 9 naranjas cada una y regala 3. ¿Cuántas naranjas le quedan?', 'respuesta': 762}


**2. Juez**

* extrae la expresion de la respuesta del modelo,la calcula y la compara con la respuesta correcta
* por seguridad solo acepta números, operadores y parentesis: nunca se ejecuta código arbitrario
* tambien exige al menos una operación: un número suelto como 41 no cuenta
* queremos que el modelo plantee el calculo, no que adivine el resultado

Se permiten dígitos, los cuatro operadores, paréntesis, punto decimal y espacio, nada de letras

In [7]:
import re
permitidos = set("0123456789+-*/(). ")

In [8]:
def extraer_expresion(texto):
    # si viene en un bloque ```...```, tomamos lo de adentro
    m = re.search(r"```(?:python)?\s*(.*?)```", texto, re.S)
    texto = m.group(1) if m else texto
    linea = next((l.strip() for l in texto.splitlines() if l.strip()), "")
    linea = re.sub(r"^print\((.*)\)$", r"\1", linea)  # print(4*12-7) -> 4*12-7
    # "respuesta = 4*12-7" o "4*12-7 = 41", nos quedamos con la parte que es una expresion
    partes = [p.strip() for p in linea.split("=") if p.strip() and set(p.strip()) <= permitidos and re.search(r"\d", p)]
    return partes[0] if partes else linea

def es_correcta(texto, respuesta):
    expresion = extraer_expresion(texto)
    # si la expresión tiene cualquier carácter fuera de la lista permitida se rechaza
    if not expresion or not set(expresion) <= permitidos:
        return False

    # exige al menos el patrón dígito -> operador -> dígito, con paréntesis y
    # espacios opcionales en medio, así se rechaza un número suelto como 41
    if not re.search(r"\d\s*\)?\s*[-+*/]\s*\(?\s*\d", expresion):
        return False
    # Compara con tolerancia 1e-6 porque una división / puede dar decimales (10/2 = 5.0)
    # Cualquier error (sintaxis inválida, división por cero) cuenta como incorrecto
    try:
        return abs(eval(expresion, {"__builtins__": {}}) - respuesta) < 1e-6
    except Exception:
        return False

probamos el juez a mano

In [9]:
for texto in ["4 * 12 - 7", "4 + 12 - 7", "print(4*12-7)", "respuesta = 4*12 - 7", "41", "La respuesta es 41", "__import__('os')"]:
    print(f"{texto} -> {es_correcta(texto, 41)}")

4 * 12 - 7 -> True
4 + 12 - 7 -> False
print(4*12-7) -> True
respuesta = 4*12 - 7 -> True
41 -> False
La respuesta es 41 -> False
__import__('os') -> False


**3. Modelo**

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

In [11]:
modelo_id = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

In [12]:
random.seed(0)
torch.manual_seed(0)

In [14]:
tok = AutoTokenizer.from_pretrained(modelo_id)
tok.padding_side = "left" # para generar en lote, en un modelo causal (que genera hacia la derecha), el relleno debe ir a la izquierda
model = AutoModelForCausalLM.from_pretrained(modelo_id, dtype=torch.float32).cuda()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [15]:
lora = LoraConfig(r=16,
                  lora_alpha=2 * 16,
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                  task_type="CAUSAL_LM")

model = get_peft_model(model, lora)  # lora arranca en cero, el modelo es identico al original
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [16]:
def hacer_prompt(pregunta):
    # One shot, con una estructura distinta a las plantillas
    mensajes = [{"role": "system", "content": "Responde solo con una expresión aritmética de Python que calcule la respuesta, usando los números del problema. Sin texto ni explicaciones."},
                {"role": "user", "content": "Un taxi cobra 5 bolivianos de base más 3 bolivianos por kilómetro. ¿Cuánto cuesta un viaje de 7 kilómetros?"},
                {"role": "assistant", "content": "5 + 3 * 7"},
                {"role": "user", "content": pregunta}]
    return tok.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)

def generar(datos, n=1, azar=False, lote=8):
    # n respuestas por problema, generadas en lotes
    model.eval()
    salidas = []
    for i in range(0, len(datos), lote):
        enc = tok([hacer_prompt(d["pregunta"]) for d in datos[i:i + lote]], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=40, do_sample=azar, temperature=0.8 if azar else None,
                                 top_p=None, top_k=None, num_return_sequences=n, pad_token_id=tok.pad_token_id)
        textos = tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        salidas += [textos[j:j + n] for j in range(0, len(textos), n)]
    return salidas

**4. Rechazo: un problema, 8 intentos**

* antes de entrenar miremos que pasa con un solo problema de un tipo difícil para el modelo
* el azar hace que cada intento sea distinto: algunos aciertan y otros no

In [17]:
p = next(d for d in entrenamiento if d["tipo"] == "lectura")
print(p["pregunta"], "| respuesta:", p["respuesta"], "\n")
for intento in generar([p], n=8, azar=True)[0]:
    print("OK" if es_correcta(intento, p["respuesta"]) else "NO", extraer_expresion(intento))

María lee 25 páginas por día durante 9 días y le faltan 9 páginas para terminar. ¿Cuántas páginas tiene el libro? | respuesta: 234 

NO (25 * 9) - 9
NO 25 - (9 * 2)
NO 40
NO 25 * 9 - 9
NO 25 - 9
NO 25 * 9
NO 25 * 9
NO 9 + 9


RFT se queda con los OK y descarta los NO

**5. Prueba, 1 intento vs 8 intentos**

* **exactitud:** el modelo responde una sola vez, sin azar (así se usa en producción)
* **pass@8:** el problema cuenta como resuelto si al menos uno de 8 intentos acierta


* si pass@8 es mucho mayor que la exactitud, el modelo ya sabe resolver esos problemas solo que no siempre elige bien
* ese espacio entre los dos números es lo que RFT aprovecha

In [18]:
# exactitud: un solo intento
def exactitud(datos):
    return sum(es_correcta(r[0], d["respuesta"]) for r, d in zip(generar(datos), datos)) / len(datos)

# 8 intentos con azar, el problema cuenta como resuelto si al menos uno acierta (any)
# mide lo que el modelo puede resolver aunque no siempre lo elija
def pass_at_n(datos, n):
    intentos = generar(datos, n=n, azar=True)
    return sum(any(es_correcta(t, d["respuesta"]) for t in ts) for ts, d in zip(intentos, datos)) / len(datos)

In [19]:
historial = [{"ronda": 0, "exactitud": exactitud(prueba), "pass@8": pass_at_n(prueba, 8)}]
print(f"modelo original, exactitud: {historial[0]['exactitud']:.0%}, pass@8: {historial[0]['pass@8']:.0%}")

modelo original, exactitud: 70%, pass@8: 80%


**6. Una ronda de RFT**

**Muestrear:** 8 intentos por problema de entrenamiento

**Rechazar:** nos quedamos con los correctos, sin repetidos y como máximo 2 por problema, para que los faciles no dominen

**Entrenar:** fine-tuning supervisado con la entropía cruzada, la perdida solo cuenta los tokens de la respuesta: el prompt se marca con -100

In [20]:
def muestrear_y_rechazar():
    ejemplos, resueltos = [], 0
    for d, intentos in zip(entrenamiento, generar(entrenamiento, n=8, azar=True)):
        # dict.fromkeys(intentos): elimina repetidos manteniendo el orden
        # si el modelo generó 85 * 9 - 3 cinco veces, cuenta una sola
        # como máximo 2 por problema
        # guarda pares (prompt, respuesta_correcta), este es el dataset supervisado
        buenos = [t for t in dict.fromkeys(intentos) if es_correcta(t, d["respuesta"])][:2]
        resueltos += bool(buenos)
        ejemplos += [(hacer_prompt(d["pregunta"]), b) for b in buenos]
    print(f"  problemas con al menos un intento correcto: {resueltos}/{len(entrenamiento)} -> {len(ejemplos)} ejemplos")
    return ejemplos

def armar_lote(ejemplos):
    # prompt + respuesta; etiquetas = -100 en el prompt y en el relleno
    ids, etiquetas = [], []
    for prompt, respuesta in ejemplos:
        # a = "tengo 3 manzanas y vendi 1, cuanto me queda?"
        a = tok(prompt, add_special_tokens=False).input_ids
        # b = "3 - 1 <eos>"
        b = tok(respuesta + tok.eos_token, add_special_tokens=False).input_ids
        #a + b = "tengo 3 manzanas y vendi 1, cuanto me queda? 3 - 1 <eos>"
        ids.append(a + b)

        # Cross Entropy Loss ignora el valor −100, la perdida se calcula solo sobre los tokens
        # de la respuesta, no queremos que el modelo aprenda a generar el prompt de sistema o
        # la pregunta, solo a responder

        # [-100] * len(a) + b = -100 -100 -100 .......-100 3 - 1 <eos>
        etiquetas.append([-100] * len(a) + b)

    largo = max(map(len, ids))

    rellenar = lambda x, v: x + [v] * (largo - len(x))
    return {"input_ids": torch.tensor([rellenar(x, tok.pad_token_id) for x in ids]).cuda(),
            "attention_mask": torch.tensor([rellenar([1] * len(x), 0) for x in ids]).cuda(),
            "labels": torch.tensor([rellenar(e, -100) for e in etiquetas]).cuda()}


# solo recibe los parámetros con requires_grad=True,
# es decir los de LoRA, los originales están congelados
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)

def entrenar(ejemplos):
    model.train()
    for epoca in range(2):
        random.shuffle(ejemplos)
        perdidas = []
        for i in range(0, len(ejemplos), 2):
            #va de dos en dos
            #perdida = model(inputs = prompt+respuesta, attention_mask = unos donde no hay padding, labels = respuesta).loss
            perdida = model(**armar_lote(ejemplos[i:i +2])).loss   # entropía cruzada
            opt.zero_grad()
            perdida.backward()
            opt.step()
            perdidas.append(perdida.item())
        print(f"  época {epoca} | pérdida {sum(perdidas) / len(perdidas):.3f}")

**7. Entrenar en rondas**

En cada ronda el modelo mejorado vuelve a intentar los problemas, así que puede resolver algunos que antes fallaba (expert iteration)

In [21]:
for ronda in range(1, 3 + 1):
    print(f"ronda {ronda}")
    ejemplos = muestrear_y_rechazar()
    assert ejemplos, "ningún intento correcto: revisar el juez o subir 8"
    entrenar(ejemplos)
    historial.append({"ronda": ronda, "exactitud": exactitud(prueba), "pass@8": pass_at_n(prueba, 8)})
    print(f"  exactitud: {historial[-1]['exactitud']:.0%} | pass@8: {historial[-1]['pass@8']:.0%}")

ronda 1
  problemas con al menos un intento correcto: 63/79 -> 95 ejemplos
  época 0 | pérdida 0.187
  época 1 | pérdida 0.143
  exactitud: 93% | pass@8: 100%
ronda 2
  problemas con al menos un intento correcto: 79/79 -> 141 ejemplos
  época 0 | pérdida 0.179
  época 1 | pérdida 0.154
  exactitud: 100% | pass@8: 100%
ronda 3
  problemas con al menos un intento correcto: 79/79 -> 156 ejemplos
  época 0 | pérdida 0.190
  época 1 | pérdida 0.160
  exactitud: 100% | pass@8: 100%


**8. Antes y después**

```
 "combustible": ("Un bus sale con {a} litros de combustible y gasta {b} litros por hora durante {c} horas. ¿Cuántos litros le quedan?",
                    lambda a, b, c: a - b * c),

```

In [23]:
import pandas as pd

with model.disable_adapter():#  sin lora = modelo original
    antes = generar(prueba)
despues = generar(prueba)

filas = [{"tipo": d["tipo"],
          "original": es_correcta(a[0], d["respuesta"]),
          "rft": es_correcta(b[0], d["respuesta"])}
         for d, a, b in zip(prueba, antes, despues)]

print(pd.DataFrame(filas).groupby("tipo").mean().map(lambda x: f"{x:.0%}"))

                     original   rft
tipo                               
combustible              100%  100%
dinero                   100%  100%
lectura                    0%  100%
multiplicar y restar      60%  100%
restar y sumar           100%  100%
sillas                    60%  100%


In [24]:
for d, a, b in list(zip(prueba, antes, despues))[:4]:
    print(d["pregunta"], "| respuesta:", d["respuesta"])
    print("- original:", extraer_expresion(a[0]), "OK" if es_correcta(a[0], d["respuesta"]) else "NO")
    print("- rft:", extraer_expresion(b[0]), "OK" if es_correcta(b[0], d["respuesta"]) else "NO")

Ana compra 85 cajas con 9 naranjas cada una y regala 3. ¿Cuántas naranjas le quedan? | respuesta: 762
- original: 85 * 9 - 3 OK
- rft: 85 * 9 - 3 OK
Ana compra 71 cajas con 2 naranjas cada una y regala 9. ¿Cuántas naranjas le quedan? | respuesta: 133
- original: 71 - 2 * 9 NO
- rft: 71 * 2 - 9 OK
Ana compra 32 cajas con 3 naranjas cada una y regala 7. ¿Cuántas naranjas le quedan? | respuesta: 89
- original: 32 * 3 - 7 OK
- rft: 32 * 3 - 7 OK
Ana compra 58 cajas con 9 naranjas cada una y regala 7. ¿Cuántas naranjas le quedan? | respuesta: 515
- original: 58 - 9 * 7 NO
- rft: 9 * 58 - 7 OK
